In [1]:
!pip install -q transformers datasets scikit-learn librosa soundfile

import os
import random
import numpy as np
import librosa
import soundfile as sf
import pandas as pd
from tqdm import tqdm
from sklearn.model_selection import StratifiedGroupKFold

np.random.seed(42)
random.seed(42)

dataset_path = '/kaggle/input/competitions/jan-2026-dl-gen-ai-project/messy_mashup/genres_stems'
esc_path = '/kaggle/input/competitions/jan-2026-dl-gen-ai-project/messy_mashup/ESC-50-master/audio'
output_dir = './resnet_wav_data'
output_csv = 'resnet_metadata.csv'

os.makedirs(output_dir, exist_ok=True)

sr = 16000
chunk_len = 114176

noise_files = [f for f in os.listdir(esc_path) if f.endswith('.wav')]
genres = sorted([d for d in os.listdir(dataset_path) if os.path.isdir(os.path.join(dataset_path, d))])
stems = ['drums.wav', 'vocals.wav', 'bass.wav', 'other.wav']


In [2]:
data_rows = []
mashup_id = 0

for genre in tqdm(genres, desc='Genres'):
    genre_path = os.path.join(dataset_path, genre)
    songs = [s for s in os.listdir(genre_path) if os.path.isdir(os.path.join(genre_path, s))]
    
    for song in songs:
        pool = [s for s in songs if s != song]
        if len(pool) == 0:
            pool = songs
            
        for _ in range(10):
            sources = random.sample(pool, 4)
            audio = np.zeros(chunk_len, dtype=np.float32)
            valid = True
            
            for i in range(4):
                stem_file = stems[i]
                src_song = sources[i]
                path = os.path.join(genre_path, src_song, stem_file)
                
                if not os.path.exists(path):
                    valid = False
                    break
                    
                y, _ = librosa.load(path, sr=sr)
                
                if len(y) > chunk_len:
                    start = random.randint(0, len(y) - chunk_len)
                    y = y[start:start + chunk_len]
                else:
                    y = np.pad(y, (0, chunk_len - len(y)))
                    
                audio = audio + y
                
            if not valid:
                continue
                
            noise_name = random.choice(noise_files)
            ny, _ = librosa.load(os.path.join(esc_path, noise_name), sr=sr)
            
            if len(ny) < chunk_len:
                ny = np.tile(ny, int(np.ceil(chunk_len / len(ny))))
            ny = ny[:chunk_len]
            
            level = random.uniform(0.05, 0.5)
            audio = audio + ny * level
            
            max_val = np.max(np.abs(audio))
            if max_val > 0:
                audio = audio / max_val
                
            filename = f"resnet_mashup_{mashup_id}.wav"
            out_path = os.path.join(output_dir, filename)
            
            sf.write(out_path, audio, sr)
            
            data_rows.append({
                'id': mashup_id,
                'genre': genre,
                'song': song,
                'filename': filename,
                'path': out_path
            })
            
            mashup_id = mashup_id + 1

df = pd.DataFrame(data_rows)
df.to_csv('original_' + output_csv, index=False)


Genres: 100%|██████████| 10/10 [49:25<00:00, 296.58s/it]


In [3]:
df['fold'] = -1
sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)

for fold_idx, (train_idx, val_idx) in enumerate(sgkf.split(df, df['genre'], df['song'])):
    df.loc[val_idx, 'fold'] = fold_idx
    
df.to_csv(output_csv, index=False)
print("ResNet dataset generation complete!")
print("Saved files:", len(df))


ResNet dataset generation complete!
Saved files: 10000
